
# UD4 — Redes Neuronales con PyTorch  
## Clasificación binaria: **Zapato vs No-zapato** (Fashion-MNIST)

En este notebook implementamos **el mismo problema binario** que ya resolvimos con Keras,
pero ahora usando **PyTorch**.

### Objetivo
- Entender explícitamente:
  - forward pass
  - backward pass
  - actualización de pesos


# Introducción a PyTorch

PyTorch permite mayor control sobre el entrenamiento de una red neuronal.
Aquí veremos explícitamente:

1. Definición del modelo
2. Forward pass
3. Cálculo de pérdida
4. Backward pass (`loss.backward()`)
5. Actualización de parámetros (`optimizer.step()`)

Es el mismo problema que en Keras para poder comparar enfoques.


## Problema a resolver

Trabajaremos con **Fashion-MNIST binario** para clasificar imágenes en dos clases.

- Entrada: imagen 28x28
- Salida: clase binaria
- Pérdida: Binary Crossentropy (o equivalente en logits según implementación)

El objetivo es comprender el ciclo completo de entrenamiento en PyTorch.


## Mini-guia PyTorch (resumen docente)

### Qué es PyTorch
PyTorch es un framework de deep learning con control explícito del entrenamiento.

Flujo típico:
1. Definir red (`nn.Module`)
2. Hacer `forward`
3. Calcular pérdida
4. Backward (`loss.backward()`)
5. Actualizar parámetros (`optimizer.step()`)

### Componentes clave
- `torch.Tensor`
- `nn.Module` (modelo)
- `nn` (capas y pérdidas)
- `optim` (optimizadores)
- `DataLoader` (batches)

### Capas frecuentes
- `nn.Linear`
- `nn.Conv2d`
- `nn.Dropout`
- `nn.BatchNorm1d/2d`

### Activaciones
- Ocultas: ReLU
- Salida binaria: Sigmoid (o logits + BCEWithLogitsLoss)
- Multiclase: logits + CrossEntropyLoss

### Regularización y estabilidad
- Dropout
- Weight decay (L2)
- BatchNorm
- Early stopping (manual)

### Regla práctica de entrenamiento
- `model.train()` para entrenar
- `model.eval()` para evaluar
- `with torch.no_grad()` en validación/inferencia



## Entorno de ejecución (CPU/GPU)

Este notebook está preparado para ejecutarse en:

- CPU local
- GPU local (si está configurada)
- Google Colab (recomendado si no hay GPU local)

Recomendación docente:
- Primera ejecución en CPU para entender el flujo.
- Segunda ejecución en GPU para comparar tiempos.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU CUDA detectada:", torch.cuda.get_device_name(0))
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("GPU MPS detectada (Apple Silicon)")
else:
    print("No se detecta GPU. Se usará CPU.")



## 1. Carga del dataset


In [ ]:

from torchvision import datasets
import numpy as np

# Cargar FashionMNIST con torchvision (devuelve PIL.Image) y convertir a arrays NumPy
train_raw = datasets.FashionMNIST(root='data', train=True, download=True)
test_raw = datasets.FashionMNIST(root='data', train=False, download=True)

x_train = np.array([np.array(img) for img, _ in train_raw])
y_train = np.array([label for _, label in train_raw])

x_test = np.array([np.array(img) for img, _ in test_raw])
y_test = np.array([label for _, label in test_raw])

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)



## 2. Construcción del problema binario


In [ ]:

shoe_classes = [5, 7, 9]

y_train_bin = np.isin(y_train, shoe_classes).astype(np.float32)
y_test_bin = np.isin(y_test, shoe_classes).astype(np.float32)



## 3. Preprocesado


In [ ]:

x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

x_train = x_train.reshape(-1, 28*28)
x_test = x_test.reshape(-1, 28*28)

X_train = torch.tensor(x_train)
y_train_t = torch.tensor(y_train_bin).unsqueeze(1)
X_test = torch.tensor(x_test)
y_test_t = torch.tensor(y_test_bin).unsqueeze(1)



## 4. Dataset y DataLoader


Explicación sobre `Dataset` y `DataLoader`:

- `Dataset`: abstracción que representa un conjunto de muestras y etiquetas. Puede ser `TensorDataset` si ya tienes tensores, o una subclase personalizada de `torch.utils.data.Dataset` que implemente `__len__` y `__getitem__` para cargar datos bajo demanda.
- `DataLoader`: itera sobre un `Dataset` y produce batches. Parámetros importantes: `batch_size`, `shuffle`, `num_workers`, `pin_memory`, y `collate_fn`.

En este notebook usamos `TensorDataset` para simplicidad. En problemas con imágenes reales conviene usar `torchvision.datasets` con `transforms` y `num_workers>0` para acelerar la carga.


In [ ]:

train_ds = TensorDataset(X_train, y_train_t)
test_ds = TensorDataset(X_test, y_test_t)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128)


### Ejemplo: `Dataset` personalizado y `collate_fn`

A continuación mostramos un `Dataset` sencillo que acepta arrays NumPy y una `collate_fn` personalizada que construye batches. Esto es útil cuando quieres controlar la conversión de tipos, padding o estructura del batch.


In [ ]:
from torch.utils.data import Dataset

class NumpyDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x, y

def default_collate(batch):
    # batch: lista de (x, y)
    xs = np.stack([b[0] for b in batch], axis=0)
    xs = torch.tensor(xs, dtype=torch.float32)
    ys = torch.tensor([b[1] for b in batch], dtype=torch.float32).unsqueeze(1)
    return xs, ys

# Uso
custom_ds = NumpyDataset(x_train, y_train_bin)
custom_loader = DataLoader(custom_ds, batch_size=128, shuffle=True, collate_fn=default_collate)

# reemplaza el train_loader original por el personalizado si quieres probarlo
train_loader = custom_loader


#### Ejemplo: `collate_fn` con padding (secuencias)

Si trabajas con secuencias de distinto tamaño (p. ej. texto), puedes usar un `collate_fn` que haga padding al batch. El siguiente ejemplo muestra la idea sobre arrays 1D:


In [ ]:
def pad_collate(batch):
    # batch: lista de (seq, label), seq es array 1D variable
    xs = [torch.tensor(b[0], dtype=torch.float32) for b in batch]
    lengths = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    maxlen = int(lengths.max())
    padded = torch.zeros(len(xs), maxlen, dtype=torch.float32)
    for i, x in enumerate(xs):
        padded[i, : x.size(0)] = x
    ys = torch.tensor([b[1] for b in batch], dtype=torch.float32).unsqueeze(1)
    return padded, lengths, ys

# Prueba rápida con datos sintéticos
import numpy as np
synthetic = [(np.random.rand(np.random.randint(3,8)), np.random.randint(0,2)) for _ in range(4)]
padded_x, lengths, y = pad_collate(synthetic)
print('padded_x.shape=', padded_x.shape, 'lengths=', lengths, 'y.shape=', y.shape)



## 5. Definición del modelo


In [ ]:

class BinaryNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

model = BinaryNet()
model



## 6. Función de pérdida y optimizador


In [ ]:

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



## 7. Bucle de entrenamiento


In [ ]:

epochs = 10
train_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")



## 8. Evaluación


In [ ]:

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        preds = (outputs > 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.size(0)

print("Test accuracy:", correct / total)



## 9. Curva de pérdida


In [ ]:

plt.plot(train_losses)
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()



## 10. Conclusiones

- PyTorch requiere más código
- El entrenamiento es explícito
- El flujo forward → backward → update es visible

En el siguiente notebook:
👉 **PyTorch multiclase**
